# Imports

In [ ]:
# Bibliothèques utilisées dans ce notebook
# ------------------------------------------------------------
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.sparse import hstack

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,f1_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


## BASELINE : Machine Learning

In [ ]:
# Chargement du dataset préparé et des colonnes requises
# ------------------------------------------------------------
df = pd.read_csv("./csv/train_text_preprocessed.csv")

colonnes_requises = [
    "productid", "imageid", "prdtypecode",
    "texte_complet", "texte_traduit",
    "texte_vectorisation", "texte_lemmatise",
]


# La même séparation sera réutilisée dans tous les notebooks du projet, donc on crée le csv une fois, s'il existe déjà on le recharge
# ------------------------------------------------------------

chemin_split = Path("./csv/splits_rakuten.csv")

if chemin_split.exists(): # Si le fichier existe on le charge
    
    splits = pd.read_csv(chemin_split)

    df = df.merge(splits[["productid", "imageid", "prdtypecode", "split"]],  on=["productid", "imageid", "prdtypecode"],how="left", validate="one_to_one")
    assert len(df) == len(splits), "Le nombre d'observations a changé."
    assert df["split"].notna().all(), "Les identifiants du dataset ne correspondent pas au split sauvegardé."
    assert set(df["split"].unique()) == {"train", "val", "test"}
    print("Decoupage existant charge :", chemin_split)

else: # Si le fichier n'existe pas, on le créé
    # Deux fiches dont le texte_traduit est identique doivent appartenir au même ensemble pour éviter le leakage.
    # ------------------------------------------------------------
    groupes = df["texte_traduit"].fillna("").str.strip().copy()

    # Les textes vides forment chacun un groupe independant
    vides = groupes.eq("")
    groupes.loc[vides] = "__vide_" + df.loc[vides, "productid"].astype(str)

    # Decoupage stratifié en 10 groupes de validation (folds)
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)

    folds = list(sgkf.split(df, df["prdtypecode"], groupes))

    # Un fold pour la validation, un pour le test
    idx_val = folds[0][1]
    idx_test = folds[1][1]

    # Les huit autres folds pour l'entraînement
    idx_train = np.setdiff1d(np.arange(len(df)), np.concatenate([idx_val, idx_test]))

    df["split"] = "train"
    df.iloc[idx_val, df.columns.get_loc("split")] = "val"
    df.iloc[idx_test, df.columns.get_loc("split")] = "test"

    df[["productid", "imageid", "prdtypecode", "split"]].to_csv(chemin_split, index=False)

    print("Nouveau decoupage cree et sauvegarde :", chemin_split)

# Trois DataFrames distincts et definitifs
train_df = df.loc[df["split"] == "train"].copy()
val_df = df.loc[df["split"] == "val"].copy()
test_df = df.loc[df["split"] == "test"].copy()  # Ne pas utiliser pour choisir les modeles

print("\nEffectifs :")
print(df["split"].value_counts())
print("\nRepartition (%) :")
print((df["split"].value_counts(normalize=True) * 100).round(2))
print("\nNombre de classes par ensemble :")
print(df.groupby("split")["prdtypecode"].nunique())


Nouveau decoupage cree et sauvegarde : csv\splits_rakuten.csv

Effectifs :
split
train    67931
test      8493
val       8492
Name: count, dtype: int64

Repartition (%) :
split
train    80.0
test     10.0
val      10.0
Name: proportion, dtype: float64

Nombre de classes par ensemble :
split
test     27
train    27
val      27
Name: prdtypecode, dtype: int64


### Etape 1  : comparer plusieurs modèles avec leurs paramètres par défaut + TF-IDF unigrams et bigrams

In [36]:
# Sélection du texte lemmatisé — uniquement train et validation
# ------------------------------------------------------------
X_train_text = train_df["texte_lemmatise"].fillna("")
X_val_text = val_df["texte_lemmatise"].fillna("")
y_train = train_df["prdtypecode"]
y_val = val_df["prdtypecode"]

# Vectorisation TF-IDF mots et bigrammes
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=50000,     # limite la taille du vocabulaire
    ngram_range=(1, 2),      # unigrams + bigrams
    min_df=2,                # ignore les mots présents dans moins de 2 documents
    max_df=0.95,             # ignore les mots présents dans plus de 95% des documents
    sublinear_tf=True        # réduit l'impact des mots très répétés dans un même texte
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)

print("Shape train TF-IDF :", X_train_tfidf.shape)
print("Shape validation TF-IDF :", X_val_tfidf.shape)
print("Nombre de features TF-IDF :", len(tfidf.get_feature_names_out()))

# 1er modèles
# ------------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,      # Nombre maximal d'itérations autorisées pour atteindre la convergence
        random_state=42     # Graine aléatoire pour rendre les résultats reproductibles
    ),

    "LinearSVC": LinearSVC(
        random_state=42     # Graine aléatoire pour rendre les résultats reproductibles
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=30,       # Profondeur maximale de l'arbre pour limiter le surapprentissage
        random_state=42     # Graine aléatoire pour rendre les résultats reproductibles
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,   # Nombre d'arbres construits dans la forêt
        n_jobs=-1,          # Utilise tous les cœurs du processeur disponibles
        random_state=42     # Graine aléatoire pour rendre les résultats reproductibles
    ),


    "SGD Classifier": SGDClassifier(
        loss="hinge",       # Fonction de perte utilisée par les SVM linéaires : pénalise les erreurs
                            # et les observations situées trop près de la frontière de décision
        random_state=42     # Graine aléatoire pour rendre les résultats reproductibles
    ),

    "Complement Naive Bayes": ComplementNB()
}

# Entraînement et évaluation des modèles + stockage des résultats
# ------------------------------------------------------------

results = []

for name, model in models.items():

    print(f"Entraînement : {name}...")

    start = time.time()

    # Entraînement
    model.fit(X_train_tfidf, y_train)

    # Prédiction sur le jeu de validation
    y_pred = model.predict(X_val_tfidf)

    elapsed_time = time.time() - start

    # Stockage des résultats
    results.append({
        "Modèle": name,
        "Représentation": "Mots + bigrammes",
        "Accuracy": accuracy_score(y_val, y_pred),
        "F1 macro": f1_score(y_val, y_pred, average="macro"),
        "F1 weighted": f1_score(y_val, y_pred, average="weighted"),
        "Temps fit + predict (s)": elapsed_time,
    })


# Tableau comparatif
# ------------------------------------------------------------

results_df = pd.DataFrame(results).sort_values("F1 macro", ascending=False).reset_index(drop=True)

display(results_df.round(4))

Shape train TF-IDF : (67931, 50000)
Shape validation TF-IDF : (8492, 50000)
Nombre de features TF-IDF : 50000
Entraînement : Logistic Regression...
Entraînement : LinearSVC...
Entraînement : Decision Tree...
Entraînement : Random Forest...
Entraînement : SGD Classifier...
Entraînement : Complement Naive Bayes...


,Modèle,Représentation,Accuracy,F1 macro,F1 weighted,Temps fit + predict (s)
0,LinearSVC,Mots + bigrammes,0.8092,0.7975,0.8082,17.1299
1,Logistic Regression,Mots + bigrammes,0.7976,0.7877,0.7976,55.1113
2,SGD Classifier,Mots + bigrammes,0.7841,0.7642,0.7738,3.3027
3,Random Forest,Mots + bigrammes,0.7761,0.7628,0.7729,265.1652
4,Complement Naive Bayes,Mots + bigrammes,0.7468,0.7147,0.7322,0.1621
5,Decision Tree,Mots + bigrammes,0.5250,0.5179,0.5581,24.6697


### Etape 2  : Essai d'entrainement du meilleur modèle sur TF-IDF caractère

In [37]:
# Vectorisation TF-IDF caractères
# ------------------------------------------------------------

tfidf_char = TfidfVectorizer(
    analyzer="char_wb",      # Analyse des séquences de caractères à l'intérieur des mots
    ngram_range=(3, 5),      # Séquences de 3 à 5 caractères
    min_df=2,                # Ignore les séquences présentes dans moins de 2 textes
    max_features=50_000,     # Maximum de 50 000 caractéristiques
    sublinear_tf=True        # Réduit l'influence des répétitions très nombreuses
)

X_train_char = tfidf_char.fit_transform(X_train_text)
X_val_char = tfidf_char.transform(X_val_text)

# LinearSVC avec uniquement les caractéristiques caractères
# ------------------------------------------------------------
start = time.time()
model_svc_char = LinearSVC(random_state=42)
model_svc_char.fit(X_train_char, y_train)
y_pred_char = model_svc_char.predict(X_val_char)

duree = time.time() - start
results.append({
    "Modèle": "LinearSVC",
    "Représentation": "Caractères 3-5 (lemmatisé)",
    "Accuracy": accuracy_score(y_val, y_pred_char),
    "F1 macro": f1_score(y_val, y_pred_char, average="macro"),
    "F1 weighted": f1_score(y_val, y_pred_char, average="weighted"),
    "Temps fit + predict (s)": duree,
})

print("Accuracy :", round(accuracy_score(y_val, y_pred_char), 4))
print("F1 macro :", round(f1_score(y_val, y_pred_char, average="macro"), 4))
print("F1 weighted :", round(f1_score(y_val, y_pred_char, average="weighted"), 4))
print("Temps :", round(duree, 2), "s")

Accuracy : 0.8145
F1 macro : 0.8043
F1 weighted : 0.813
Temps : 91.28 s


### Etape 3  : Combinaison des matrices mots + caractère pour l'entrainement

In [38]:
# Combinaison TF-IDF mots + caractères
# ------------------------------------------------------------

X_train_combined = hstack([X_train_tfidf, X_train_char]).tocsr()

X_val_combined = hstack([X_val_tfidf, X_val_char]).tocsr()

print("Shape train :", X_train_combined.shape)
print("Shape validation :", X_val_combined.shape)

# LinearSVC sur 100 000 features
# ------------------------------------------------------------
start = time.time()
model_svc_combined = LinearSVC(random_state=42)
model_svc_combined.fit(X_train_combined, y_train)
y_pred_combined = model_svc_combined.predict(X_val_combined)

duree = time.time() - start
results.append({
    "Modèle": "LinearSVC",
    "Représentation": "Mots + caractères (lemmatisé)",
    "Accuracy": accuracy_score(y_val, y_pred_combined),
    "F1 macro": f1_score(y_val, y_pred_combined, average="macro"),
    "F1 weighted": f1_score(y_val, y_pred_combined, average="weighted"),
    "Temps fit + predict (s)": duree,
})

print("Accuracy :", round(accuracy_score(y_val, y_pred_combined), 4))
print("F1 macro :", round(f1_score(y_val, y_pred_combined, average="macro"), 4))
print("F1 weighted :", round(f1_score(y_val, y_pred_combined, average="weighted"), 4))
print("Temps :", round(duree, 2), "s")

Shape train : (67931, 100000)
Shape validation : (8492, 100000)
Accuracy : 0.8183
F1 macro : 0.807
F1 weighted : 0.8175
Temps : 33.5 s


### Etape 4  : Test d'entraînement du texte sans lemmatisation pour voir si les résultats sont meilleurs

In [39]:
# Nouvelle représentation lexicale sans lemmatisation
# ------------------------------------------------------------
X_train_vec = train_df["texte_vectorisation"].fillna("")
X_val_vec = val_df["texte_vectorisation"].fillna("")

# TF-IDF mots — mêmes paramètres que l'étape 1
# ------------------------------------------------------------
tfidf_vec = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

# TF-IDF caractères — mêmes paramètres que l'étape 2
# ------------------------------------------------------------
tfidf_vec_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
)

X_train_vec_tfidf = tfidf_vec.fit_transform(X_train_vec)
X_val_vec_tfidf = tfidf_vec.transform(X_val_vec)

X_train_vec_char = tfidf_vec_char.fit_transform(X_train_vec)
X_val_vec_char = tfidf_vec_char.transform(X_val_vec)

X_train_vec_combined = hstack([X_train_vec_tfidf, X_train_vec_char]).tocsr()
X_val_vec_combined = hstack([X_val_vec_tfidf, X_val_vec_char]).tocsr()

print("Shape train :", X_train_vec_combined.shape)
print("Shape validation :", X_val_vec_combined.shape)

# Entraînement du LinearSVC sans lemmatisation
# ------------------------------------------------------------
start = time.time()
model_svc_vec_combined = LinearSVC(random_state=42)
model_svc_vec_combined.fit(X_train_vec_combined, y_train)
y_pred_vec_combined = model_svc_vec_combined.predict(X_val_vec_combined)

duree = time.time() - start
results.append({
    "Modèle": "LinearSVC",
    "Représentation": "Mots + caractères (sans lemmatisation)",
    "Accuracy": accuracy_score(y_val, y_pred_vec_combined),
    "F1 macro": f1_score(y_val, y_pred_vec_combined, average="macro"),
    "F1 weighted": f1_score(y_val, y_pred_vec_combined, average="weighted"),
    "Temps fit + predict (s)": duree,
})

print("Accuracy :", round(accuracy_score(y_val, y_pred_vec_combined), 4))
print("F1 macro :", round(f1_score(y_val, y_pred_vec_combined, average="macro"), 4))
print("F1 weighted :", round(f1_score(y_val, y_pred_vec_combined, average="weighted"), 4))
print("Temps :", round(duree, 2), "s")


Shape train : (67931, 100000)
Shape validation : (8492, 100000)
Accuracy : 0.8227
F1 macro : 0.8115
F1 weighted : 0.8219
Temps : 33.23 s


### 5. Comparaison sur la validation uniquement

In [40]:
# Bilan général des quatre étapes
# ------------------------------------------------------------
results_df = (
    pd.DataFrame(results)
    .sort_values("F1 macro", ascending=False)
    .reset_index(drop=True)
)

display(results_df.round(4))
print("Meilleure expérience sur la validation :")
print(results_df.loc[0, ["Modèle", "Représentation", "F1 macro"]].to_string())


,Modèle,Représentation,Accuracy,F1 macro,F1 weighted,Temps fit + predict (s)
0,LinearSVC,Mots + caractères (sans lemmatisation),0.8227,0.8115,0.8219,33.2334
1,LinearSVC,Mots + caractères (lemmatisé),0.8183,0.8070,0.8175,33.5001
2,LinearSVC,Caractères 3-5 (lemmatisé),0.8145,0.8043,0.8130,91.2842
3,LinearSVC,Mots + bigrammes,0.8092,0.7975,0.8082,17.1299
4,Logistic Regression,Mots + bigrammes,0.7976,0.7877,0.7976,55.1113
5,SGD Classifier,Mots + bigrammes,0.7841,0.7642,0.7738,3.3027
6,Random Forest,Mots + bigrammes,0.7761,0.7628,0.7729,265.1652
7,Complement Naive Bayes,Mots + bigrammes,0.7468,0.7147,0.7322,0.1621
8,Decision Tree,Mots + bigrammes,0.5250,0.5179,0.5581,24.6697


Meilleure expérience sur la validation :
Modèle                                         LinearSVC
Représentation    Mots + caractères (sans lemmatisation)
F1 macro                                        0.811473


rnn/lstm + bi lstm + transformers (familles BERT)

Après la partie modeling il faudra faire une partie interprétabilité. 
Texte : outils comme shap + réduction de dimention (alléger ses données avant mais risqué, sinon travailler les endings, des vecteurs qui représentent l'info donc réduction de dimension avec (PCA, TSNE) pour représenter graphiquement de facon visible)